# SAINI AI — Free GPU hardware preflight
**For Qwen3.8 27B and Gemma 4 26B-A4B research.** This notebook does not download models, train models or make paid requests. Run in a **separate** Kaggle research notebook, not in the SAINI AI Songs environment.

Use its output to decide whether either checkpoint can fit. Two 16 GB GPUs do not automatically behave like one 32 GB GPU.


In [ ]:
# GPU-only diagnostic for a separate SAINI AI research notebook.
# Does not download weights, call model APIs, run training, or create cloud jobs.
import json
import os
import platform
import shutil

report = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cwd": os.getcwd(),
    "torch": None,
    "cuda_available": False,
    "gpu_count": 0,
    "gpus": [],
}
try:
    import torch
    report["torch"] = torch.__version__
    report["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        report["gpu_count"] = torch.cuda.device_count()
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            report["gpus"].append({
                "index": i,
                "name": p.name,
                "total_vram_gib": round(p.total_memory / (1024 ** 3), 2),
                "compute_capability": f"{p.major}.{p.minor}",
            })
except Exception as exc:
    report["torch_error"] = str(exc)

for location in ("/kaggle/working", "/tmp"):
    if os.path.isdir(location):
        total, used, free = shutil.disk_usage(location)
        report["disk_" + location.replace("/", "_")] = {
            "total_gib": round(total / (1024 ** 3), 2),
            "free_gib": round(free / (1024 ** 3), 2)
        }
print(json.dumps(report, indent=2))
print("\nNEXT: check model revision, license, supported loader, VRAM, disk and quota.")
print("PRECHECK ONLY: NO MODEL DOWNLOAD, NO TRAINING, NO PAID API, NO DEPLOYMENT.")
